# Daily Challenge: BaristaBot with LangGraph + Gemini

**Course:** Developers Institute  **Week 9 - Day 5**  
**Author:** Alex Goldbaum

A conversational cafe ordering system built on **LangGraph** (stateful graph runtime)
and the **Gemini API** (via `langchain-google-genai`). The bot reads a live menu,
takes orders in natural language, confirms with the user, and places the order.

We build the graph in stages, exactly as the assignment requests, so each step is
individually testable.


## 1. Install dependencies


In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0"


## 2. Set the Google API key

The assignment uses Kaggle secrets. Since we run on Colab, we read from
Colab Secrets (left sidebar 🔑 → add `GOOGLE_API_KEY`). If you run locally,
set the env var before launching Jupyter and skip this cell.


In [ ]:
import os

if not os.environ.get('GOOGLE_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        import getpass
        os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter GOOGLE_API_KEY: ')

assert os.environ.get('GOOGLE_API_KEY'), 'GOOGLE_API_KEY is required'
print('Gemini API key configured.')


## 3. Define the state schema and system instruction

The `OrderState` carries three things between nodes: the chat history (with the
`add_messages` reducer that *appends* instead of replacing), the in-progress order,
and a `finished` flag used by the conditional edge to exit the graph.


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """State representing the customer's order conversation."""
    messages: Annotated[list, add_messages]
    order: list[str]
    finished: bool


In [ ]:
BARISTABOT_SYSINT = (
    'system',
    'You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the '
    'available products you have and you will answer any questions about menu items (and only about '
    'menu items - no off-topic discussion, but you can chat about the products and their history). '
    'The customer will place an order for 1 or more items from the menu, which you will structure '
    'and send to the ordering system after confirming the order with the human. '
    '\n\n'
    'Add items to the customer\'s order with add_to_order, and reset the order with clear_order. '
    'To see the contents of the order so far, call get_order (this is shown to you, not the user) '
    'Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will '
    'display the order items to the user and returns their response to seeing the list. Their response may contain modifications. '
    'Always verify and respond with drink and modifier names from the MENU before adding them to the order. '
    'If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. '
    'You only have the modifiers listed on the menu. '
    'Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make '
    'any necessary updates and then call place_order. Once place_order has returned, thank the user and '
    'say goodbye!',
)

WELCOME_MSG = 'Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?'


## 4. Single-turn chatbot graph

Smallest possible graph: a single `chatbot` node, entered from `START`. Each node is
a function that receives the state and returns a partial state update.


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-latest')

def chatbot(state: OrderState) -> OrderState:
    """A single-turn chatbot: feeds system prompt + history to Gemini."""
    message_history = [BARISTABOT_SYSINT] + state['messages']
    return {'messages': [llm.invoke(message_history)]}


graph_builder = StateGraph(OrderState)
graph_builder.add_node('chatbot', chatbot)
graph_builder.add_edge(START, 'chatbot')

chat_graph = graph_builder.compile()
print('Single-turn graph compiled.')


## 5. Visualize the graph

`get_graph().draw_mermaid_png()` returns a PNG that Colab renders inline.


In [ ]:
from IPython.display import Image

Image(chat_graph.get_graph().draw_mermaid_png())


## 6. Run the graph for one turn

Initial state has a single user message. After `invoke`, the state contains the
user message plus the AI reply (because of the `add_messages` reducer).


In [ ]:
user_msg = 'Hi! Do you have any tea drinks?'
state = chat_graph.invoke({'messages': [('user', user_msg)]})

for msg in state['messages']:
    print(f'{type(msg).__name__}: {msg.content}')


## 7. Manually append a second turn

We append a new user message and re-invoke. The history grows because the reducer
appends instead of replacing.


In [ ]:
user_msg = 'Great, can I get an Earl Grey then?'

state['messages'].append(('user', user_msg))
state = chat_graph.invoke(state)

for msg in state['messages']:
    print(f'{type(msg).__name__}: {msg.content}')


## 8. Add a `human` node so the loop lives inside the graph

Instead of looping in Python, we let LangGraph loop between `chatbot` and `human`.
The human node prints the last AI message and reads `input()`. We also wire a
welcome message for the first turn.


In [ ]:
from langchain_core.messages.ai import AIMessage


def human_node(state: OrderState) -> OrderState:
    """Show last model message, read user input, flag exit on quit."""
    last_msg = state['messages'][-1]
    print('Model:', last_msg.content)

    user_input = input('User: ')

    if user_input in {'q', 'quit', 'exit', 'goodbye'}:
        state['finished'] = True

    return state | {'messages': [('user', user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """Chatbot that emits the welcome message on the empty first turn."""
    if state['messages']:
        new_output = llm.invoke([BARISTABOT_SYSINT] + state['messages'])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {'messages': [new_output]}


graph_builder = StateGraph(OrderState)
graph_builder.add_node('chatbot', chatbot_with_welcome_msg)
graph_builder.add_node('human', human_node)
graph_builder.add_edge(START, 'chatbot')
graph_builder.add_edge('chatbot', 'human')
print('Graph with human node wired (still needs exit condition).')


## 9. Conditional edge — exit when the user quits

Without an exit condition the loop would never end. We add a conditional edge
from `human` that goes back to `chatbot` unless `finished` is True, in which
case it goes to `END`.


In [ ]:
from typing import Literal


def maybe_exit_human_node(state: OrderState) -> Literal['chatbot', '__end__']:
    """Route to the chatbot, unless the user is exiting."""
    if state.get('finished', False):
        return END
    else:
        return 'chatbot'


graph_builder.add_conditional_edges('human', maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

Image(chat_with_human_graph.get_graph().draw_mermaid_png())


## 10. Add the live menu as a stateless tool

We expose `get_menu` as a `@tool`. The `ToolNode` automatically calls it whenever
the model emits a tool call. After running a tool, control returns to `chatbot`
so the model can incorporate the tool result.


In [ ]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    'dirty' means add a shot of espresso to a drink that doesn't usually have it, like 'Dirty Chai Latte'.
    'Regular milk' is the same as 'whole milk'.
    'Sweetened' means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
    """


In [ ]:
from langgraph.prebuilt import ToolNode

tools = [get_menu]
tool_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: OrderState) -> Literal['tools', 'human']:
    """Route to tools if the model emitted a tool call, else to human."""
    if not (msgs := state.get('messages', [])):
        raise ValueError(f'No messages found when parsing state: {state}')
    msg = msgs[-1]
    if hasattr(msg, 'tool_calls') and len(msg.tool_calls) > 0:
        return 'tools'
    else:
        return 'human'


def chatbot_with_tools(state: OrderState) -> OrderState:
    """Chatbot bound to the tools, with welcome message on empty state."""
    defaults = {'order': [], 'finished': False}
    if state['messages']:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state['messages'])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return defaults | state | {'messages': [new_output]}


graph_builder = StateGraph(OrderState)
graph_builder.add_node('chatbot', chatbot_with_tools)
graph_builder.add_node('human', human_node)
graph_builder.add_node('tools', tool_node)

graph_builder.add_conditional_edges('chatbot', maybe_route_to_tools)
graph_builder.add_conditional_edges('human', maybe_exit_human_node)
graph_builder.add_edge('tools', 'chatbot')
graph_builder.add_edge(START, 'chatbot')

graph_with_menu = graph_builder.compile()
Image(graph_with_menu.get_graph().draw_mermaid_png())


## 11. Order-handling tools + custom `ordering` node

The order tools (`add_to_order`, `confirm_order`, `get_order`, `clear_order`,
`place_order`) are declared with `@tool` for their schema only — their bodies are
empty. The actual logic lives in `order_node`, which can mutate the state directly.
This separation is important: the model never touches internal state, only emits
tool calls that the node interprets.


In [ ]:
from collections.abc import Iterable
from random import randint

from langchain_core.messages.tool import ToolMessage


@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers."""


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct."""


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""


@tool
def clear_order():
    """Removes all items from the user's order."""


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment."""


def order_node(state: OrderState) -> OrderState:
    """Apply order-mutating tool calls to the state."""
    tool_msg = state['messages'][-1]
    order = list(state.get('order', []))
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:
        name = tool_call['name']

        if name == 'add_to_order':
            modifiers = tool_call['args'].get('modifiers', []) or []
            modifier_str = ', '.join(modifiers) if modifiers else 'no modifiers'
            order.append(f"{tool_call['args']['drink']} ({modifier_str})")
            response = '\n'.join(order)

        elif name == 'confirm_order':
            print('Your order:')
            if not order:
                print('  (no items)')
            for drink in order:
                print(f'  {drink}')
            response = input('Is this correct? ')

        elif name == 'get_order':
            response = '\n'.join(order) if order else '(no order)'

        elif name == 'clear_order':
            order.clear()
            response = 'Order cleared.'

        elif name == 'place_order':
            order_text = '\n'.join(order)
            print('Sending order to kitchen!')
            print(order_text)
            order_placed = True
            response = randint(1, 5)

        else:
            raise NotImplementedError(f"Unknown tool call: {name}")

        outbound_msgs.append(
            ToolMessage(
                content=str(response) if response is not None else '',
                name=name,
                tool_call_id=tool_call['id'],
            )
        )

    return {'messages': outbound_msgs, 'order': order, 'finished': order_placed}


In [ ]:
def maybe_route_to_tools(state: OrderState) -> str:
    """Route between tools, ordering, human or END."""
    if not (msgs := state.get('messages', [])):
        raise ValueError(f'No messages found when parsing state: {state}')
    msg = msgs[-1]

    if state.get('finished', False):
        return END

    elif hasattr(msg, 'tool_calls') and len(msg.tool_calls) > 0:
        if any(
            tool['name'] in tool_node.tools_by_name.keys() for tool in msg.tool_calls
        ):
            return 'tools'
        else:
            return 'ordering'

    else:
        return 'human'


## 12. Final graph — chatbot + tools + ordering + human

Two sets of tools: `auto_tools` (handled by `ToolNode`) and `order_tools` (handled
by our custom `order_node`). The LLM is bound to *all* of them so it can choose
any. The router sends auto tool calls to `tools` and order tool calls to `ordering`.


In [ ]:
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

llm_with_tools = llm.bind_tools(auto_tools + order_tools)


def chatbot_with_tools(state: OrderState) -> OrderState:
    """Chatbot bound to all tools, with welcome message on empty state."""
    defaults = {'order': [], 'finished': False}
    if state['messages']:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state['messages'])
    else:
        new_output = AIMessage(content=WELCOME_MSG)
    return defaults | state | {'messages': [new_output]}


graph_builder = StateGraph(OrderState)
graph_builder.add_node('chatbot', chatbot_with_tools)
graph_builder.add_node('human', human_node)
graph_builder.add_node('tools', tool_node)
graph_builder.add_node('ordering', order_node)

graph_builder.add_conditional_edges('chatbot', maybe_route_to_tools)
graph_builder.add_conditional_edges('human', maybe_exit_human_node)
graph_builder.add_edge('tools', 'chatbot')
graph_builder.add_edge('ordering', 'chatbot')
graph_builder.add_edge(START, 'chatbot')

graph_with_order_tools = graph_builder.compile()
Image(graph_with_order_tools.get_graph().draw_mermaid_png())


## 13. Run the full BaristaBot

This cell starts the interactive loop. Try:
- *"What teas do you have?"*
- *"I'll have a Latte with oat milk."*
- *"Actually make that iced."*
- After confirming, the order is placed and the graph exits.

Type `q` at any user prompt to quit early.


In [ ]:
from pprint import pprint

config = {'recursion_limit': 100}

state = graph_with_order_tools.invoke({'messages': []}, config)

print()
print('=== Final state ===')
pprint(state)


## What we built

- A LangGraph state machine with **four nodes** (`chatbot`, `human`, `tools`, `ordering`)
  and conditional routing based on whether the model emitted a tool call, what *kind*
  of tool call, and whether the user wants to quit.
- **Two classes of tools**: stateless (`get_menu`, auto-handled by `ToolNode`) and
  stateful (order operations, handled by a custom node that mutates `OrderState`).
- **`add_messages` reducer** so chat history accumulates instead of being replaced.
- **Welcome message** on the empty initial turn, **conditional exit** when the order
  is placed or the user types quit.

The same pattern scales to real workflows (customer support, internal copilots,
data-pipeline operators): replace the tools, keep the graph structure.
